# Week 6 - Assignment

### Apache Spark Fundamentals and Data Processing using PySpark

#### Objective: To understand Apache Spark architecture, lazy evaluation, DataFrame transformations, schema handling, filtering, file formats (CSV and Parquet), and performance optimization by building an end-to-end data processing pipeline using PySpark in Databricks.

##### Q1.Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

Driver:
The Driver is the main process of a Spark application. It is responsible for creating the SparkSession and SparkContext, converting user code into an execution plan, creating the Directed Acyclic Graph (DAG), scheduling jobs and tasks, communicating with the Cluster Manager, and collecting results from the Executors.

Cluster Manager:
The Cluster Manager is responsible for managing the resources required by Spark applications. It allocates CPU and memory resources, launches Executors on worker nodes, and manages the overall cluster resources. Examples include Standalone, YARN, Kubernetes, and Apache Mesos.

Executor:
An Executor is a JVM process that runs on worker nodes. It executes the tasks assigned by the Driver, performs data processing, stores cached data in memory or disk, and sends the execution results back to the Driver.

%md
##### Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

**Answer:**

**Driver:**
The Driver is the main part of a Spark application. It creates the SparkSession and SparkContext, converts the user program into an execution plan, creates the DAG (Directed Acyclic Graph), schedules jobs and tasks, communicates with the Cluster Manager, and collects the final results from the Executors.

**Cluster Manager:**
The Cluster Manager is responsible for managing the resources in the cluster. It allocates CPU and memory to the Spark application, starts Executors on worker nodes, and manages the available resources. Some commonly used cluster managers are Standalone, YARN, Kubernetes, and Mesos.

**Executor:**
An Executor is a JVM process that runs on a worker node. It executes the tasks assigned by the Driver, processes the data, stores cached data if required, and sends the results back to the Driver.

**Spark Architecture Flow:**

Application  
⬇  
Driver  
⬇  
SparkContext  
⬇  
Cluster Manager  
⬇  
Worker Nodes  
⬇  
Executors  
⬇  
Tasks

##### Q2. How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

**Answer:**

Spark uses Lazy Evaluation, which means it does not execute transformations as soon as they are written. Instead, it keeps track of all the transformations and waits until an action such as `show()`, `count()`, or `write()` is called. At that point, Spark creates an execution plan and runs the job.

Lazy Evaluation improves performance in the following ways:

- Spark looks at the complete sequence of transformations and creates an efficient execution plan.
- It avoids running unnecessary operations and computes only the required output.
- Multiple transformations are combined whenever possible, reducing the number of passes over the data.
- It minimizes disk I/O and memory usage by avoiding unnecessary intermediate results.
- Spark builds a DAG (Directed Acyclic Graph), which is divided into stages and tasks for efficient execution on executors.

Because of these optimizations, Spark can process large datasets faster and use cluster resources more efficiently.

##### Q3. Read a CSV file with header and inferSchema enabled

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/workspace/default/week6_files/source.csv")

df.show(5)

+--------+-------+----------+------------+-----------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    old_name|   category|   price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+-----------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|       Novel|      Books| 1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|      Laptop|Electronics|10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|Coffee Beans|    Grocery|13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004|      Camera|Electronic

In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: integer (nullable = true)



##### Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

**Answer:**

CSV (Comma Separated Values) is a row-based file format. In this format, all the values of a single row are stored together. Whenever Spark reads a CSV file, it generally has to scan the complete rows even if only a few columns are required.

Parquet is a columnar file format. It stores values of the same column together instead of storing complete rows. This allows Spark to read only the columns that are needed for a query.

**Difference between CSV and Parquet**

| CSV | Parquet |
|------|----------|
| Row-based storage | Column-based storage |
| Plain text format | Binary format |
| Does not store schema | Stores schema information |
| Less compression | Better compression |
| Larger file size | Smaller file size |
| Slower for analytics | Faster for analytics |

**Why Parquet performs better:**

- Spark reads only the required columns instead of the entire dataset.
- Compression reduces the file size and saves storage.
- Less data is loaded into memory, which improves processing speed.
- It supports Predicate Pushdown, allowing Spark to skip unnecessary data while reading.
- It is well suited for large-scale data processing and analytical workloads.

Therefore, Parquet is generally preferred over CSV in Spark because it provides better storage efficiency and faster query performance.

##### Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [0]:
electronics_df=df.filter(
    df.category=="Electronics"
).select(
    "product_id",
    "price"
)

electronics_df.show()

+----------+--------+
|product_id|   price|
+----------+--------+
|     P1002|10970.97|
|     P1004|22998.27|
|     P1005| 33080.1|
|     P1010|19111.88|
|     P1015|11273.63|
|     P1019|40410.71|
|     P1022|23865.59|
|     P1045|41652.91|
|     P1059|26816.41|
|     P1065|22552.66|
|     P1071|12657.32|
|     P1075|38645.12|
|     P1078|    NULL|
|     P1080|31576.49|
|     P1083| 23289.4|
|     P1091|41201.57|
|     P1092| 44368.6|
|     P1112| 8635.07|
|     P1113|43532.69|
|     P1129|35749.48|
+----------+--------+
only showing top 20 rows


###### Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [0]:
from pyspark.sql.functions import col,regexp_replace

revised_df=df.withColumnRenamed(
    "old_name",
    "new_name"
).withColumn(
    "price",
    regexp_replace(
        col("price"),
        "\\$",
        ""
    ).cast("double")
)

revised_df.printSchema()
revised_df.show(10)

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- discount_pct: integer (nullable = true)

+--------+-------+----------+------------+--------------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    new_name|      category|   price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+--------

%md
##### Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

**Answer:**

Spark uses a Lineage Graph, also called a Directed Acyclic Graph (DAG), to keep track of all the transformations performed on the data. Whenever transformations such as `filter()`, `select()`, or `groupBy()` are applied, Spark does not execute them immediately. Instead, it records these operations in the DAG and waits until an action is called.

The DAG contains the complete sequence of transformations needed to produce the final result. Since Spark knows how every partition is created, it can recover data if a failure occurs.

If a worker node or executor fails, the partitions stored on that node are lost. Spark checks the DAG to identify the transformations that were used to create those missing partitions. It then recomputes only the lost partitions from the original data and assigns the task to another available executor. This avoids reprocessing the entire dataset and saves both time and resources.

For example, if the operations are:

Read CSV → filter() → select() → groupBy() → output

and one worker node fails after the `groupBy()` operation, Spark uses the lineage information to execute the required transformations again only for the missing partition instead of restarting the whole application.

Because of this approach, Spark provides automatic fault tolerance without storing multiple copies of intermediate data. It improves reliability, reduces storage overhead, and allows Spark applications to continue processing even when worker nodes fail.

##### Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [0]:
df_orders=df.filter(
    (df.status=="Completed")&
    (df.amount>1000)
)

df_orders.show(10)

+--------+-------+----------+------------+--------------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    old_name|      category|   price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+--------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100003|  20003|     P1003|Coffee Beans|       Grocery|13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004|      Camera|   Electronics|22998.27|  22998.27|       7|160987.89|Completed|  East|     Low|  Priya Singh|2024-07-15|          15|
|  100012|   NULL|     P1012|     Toaster|Home & Kitchen|26991.98|  26991.98|       9|242927.82|Completed|  East|  Medium|   Priya Iyer|2024-09-02|           0|
|  100013|  20013|     P1013|     

##### Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

**Answer:**

Predicate Pushdown is an optimization feature used by Apache Spark when reading Parquet files. It allows Spark to apply filter conditions while reading the data instead of loading the entire dataset into memory first. This reduces the amount of data that needs to be processed.

Parquet stores data in a columnar format along with metadata such as column statistics, minimum and maximum values, and row group information. Spark uses this metadata to determine which parts of the file satisfy the filter condition. If a row group does not match the condition, Spark skips it completely without reading it.

For example, if a query filters records with `amount>1000`, Spark checks the Parquet metadata and loads only the row groups that may contain values greater than 1000. The remaining row groups are ignored.

Since only the required data is read, less data is loaded into memory, disk I/O is reduced, and network traffic is minimized. This results in faster query execution and better resource utilization.

Therefore, Predicate Pushdown is one of the major reasons why Parquet provides better performance than CSV when processing large datasets in Apache Spark.

###### Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [0]:
final_price_df=df.withColumn(
    "final_price",
    df.base_price*1.18
)

final_price_df.select(
    "product_id",
    "base_price",
    "final_price"
).show(10)

+----------+----------+------------------+
|product_id|base_price|       final_price|
+----------+----------+------------------+
|     P1001|   1299.29|         1533.1622|
|     P1002|  10970.97|12945.744599999998|
|     P1003|  13929.67|16437.010599999998|
|     P1004|  22998.27|27137.958599999998|
|     P1005|   33080.1|         39034.518|
|     P1006|  13385.54|        15794.9372|
|     P1007|  11490.95|         13559.321|
|     P1008|  44239.92|52203.105599999995|
|     P1009|  11005.07|        12985.9826|
|     P1010|  19111.88|        22552.0184|
+----------+----------+------------------+
only showing top 10 rows


%md
##### Q11. What is the difference between Transformations and Actions? Provide two examples of each.

**Answer:**

In Apache Spark, **Transformations** are operations that modify or create a new DataFrame or RDD from an existing one. They are lazily evaluated, which means Spark does not execute them immediately. Instead, it stores these operations and builds a DAG (Directed Acyclic Graph). The transformations are executed only when an action is performed.

**Examples of Transformations:**

1. `filter()` - Used to filter rows based on a given condition.  
   Example: `df.filter(df.amount>1000)`

2. `select()` - Used to select one or more required columns.  
   Example: `df.select("product_id","price")`

**Actions** are operations that trigger the execution of all the transformations. They process the data and either return a result to the Driver or save the output.

**Examples of Actions:**

1. `show()` - Displays the records of a DataFrame.  
   Example: `df.show()`

2. `count()` - Returns the total number of rows in a DataFrame.  
   Example: `df.count()`

**Example:**

`df.filter(df.amount>1000).select("product_id","amount").show()`

In this example, `filter()` and `select()` are transformations, while `show()` is an action that starts the execution.

Therefore, transformations define the processing steps, whereas actions execute those steps and produce the final output.

##### Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/week6_files/source_parquet")

In [0]:
input_df=spark.read.parquet(
    "/Volumes/workspace/default/week6_files/source_parquet"
)

clean_df=input_df.filter(
    input_df.user_id.isNotNull()
)

clean_df.write \
    .mode("overwrite") \
    .option("header","true") \
    .csv("/Volumes/workspace/default/week6_files/output_csv")

In [0]:
clean_df=df.filter(
    df.user_id.isNotNull()
)

clean_df.show(10)

+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|    old_name|      category|    price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|       Novel|         Books|  1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|      Laptop|   Electronics| 10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|Coffee Beans|       Grocery| 13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100004|  20004|     P1004

##### Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

**Answer:**

In Apache Spark, an application can run in **Client Mode** or **Cluster Mode** depending on where the Driver program is executed.

**Client Mode:**
- The Driver runs on the local machine where the Spark application is submitted.
- The client machine communicates with the Cluster Manager and Executors.
- If the client machine is disconnected or stops, the Spark application also stops.
- It is mainly used during development, testing, and debugging.

**Cluster Mode:**
- The Driver runs inside the cluster on one of the worker nodes.
- The Cluster Manager launches both the Driver and Executors.
- The application continues running even if the client disconnects after submission.
- It is mainly used in production environments because it is more reliable and efficient.

Therefore, the main difference is that in Client Mode the Driver runs on the user's machine, whereas in Cluster Mode the Driver runs inside the cluster, making it more suitable for long-running and production applications.**

###### Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [0]:
filtered_df=df.filter(
    (df.region=="North")|
    (df.priority=="High")
)

filtered_df.show(10)

+--------+-------+----------+--------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|order_id|user_id|product_id|      old_name|      category|    price|base_price|quantity|   amount|   status|region|priority|customer_name|order_date|discount_pct|
+--------+-------+----------+--------------+--------------+---------+----------+--------+---------+---------+------+--------+-------------+----------+------------+
|  100001|  20001|     P1001|         Novel|         Books|  1299.29|   1299.29|       4|  5197.16|  Pending| North|     Low|  Pooja Verma|2025-03-08|           0|
|  100002|  20002|     P1002|        Laptop|   Electronics| 10970.97|  10970.97|       1| 10970.97|  Pending|  West|    High| Rohan Sharma|2024-06-12|          15|
|  100003|  20003|     P1003|  Coffee Beans|       Grocery| 13929.67|  13929.67|       6| 83578.02|Completed| North|  Medium|  Priya Singh|2024-09-27|          20|
|  100005|  2000